# MLOps Example Notebook - Iris Classifier

This notebook demonstrates the required cell tags for the MLOps platform.
Each code cell has a `tags` entry in its metadata that the platform uses
to identify pipeline phases.

**Required tags:** `mlops:config`, `mlops:preprocessing`, `mlops:training`, `mlops:export`

**Optional tags:** `mlops:data`, `mlops:evaluation`

In [ ]:
# Papermill injected parameters (do not edit this cell manually)
# These are overwritten at runtime by the pipeline.
MODEL_OUTPUT_PATH = "./model.joblib"
PIPELINE_ID = "local-dev"
MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_RUN_ID = "local-dev"
TRAINING_FEATURES_PATH = ""  # where to write the exact frame passed to fit()
MODEL_NOTES_PATH = ""        # where to write what each input and class means

In [ ]:
# ============================================================
# mlops:config - Model metadata
# ============================================================
# Adapt MODEL_NAME and VERSION for your project.
# The platform reads these values to register the model.

MODEL_NAME = "iris-classifier"
VERSION = "2"  # v2: the model takes measurements in cm (scaler inside the model)

print(f"Model: {MODEL_NAME} v{VERSION}")
print(f"Pipeline ID: {PIPELINE_ID}")

In [ ]:
# ============================================================
# mlops:data - Data loading
# ============================================================
# Replace this with your own data loading logic.
# The platform does not require a specific data source.

from sklearn.datasets import load_iris
import pandas as pd

FEATURE_COLUMNS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

iris = load_iris()
X = pd.DataFrame(iris.data, columns=FEATURE_COLUMNS)  # measurements in cm
y = pd.Series(iris.target, name="target")

print(f"Dataset shape: {X.shape}")
print(f"Classes: {list(iris.target_names)}")

In [ ]:
# ============================================================
# mlops:preprocessing - Data preparation
# ============================================================
# Split only. The scaling is part of the model (see training), so the
# deployed model receives the same raw measurements it was trained on.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# ============================================================
# mlops:training - Model training
# ============================================================
# The scaler and the forest are one Pipeline: the saved model scales its
# own inputs, so predictions take measurements in cm.

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ("scaler", StandardScaler()),
    ("forest", RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
    )),
])

model.fit(X_train, y_train)
print("Training complete.")

In [ ]:
# ============================================================
# mlops:evaluation - Metrics and logging
# ============================================================
# Log metrics to MLflow. The platform reads 'accuracy' for
# auto-deployment decisions.

from sklearn.metrics import accuracy_score, classification_report
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

y_pred = model.predict(X_test)
score = accuracy_score(y_test, y_pred)

print(classification_report(y_test, y_pred, target_names=list(iris.target_names)))
print(f"Accuracy: {score:.4f}")

# Use the run started by the pipeline if available, otherwise start a new one
run_ctx = mlflow.start_run(run_id=MLFLOW_RUN_ID) if MLFLOW_RUN_ID != "local-dev" else mlflow.start_run()
with run_ctx:
    mlflow.log_metric("accuracy", score)
    mlflow.log_metric("n_estimators", 100)
    mlflow.log_metric("max_depth", 5)

In [ ]:
# ============================================================
# mlops:export - Save model artifact
# ============================================================
# Save the trained model to MODEL_OUTPUT_PATH.
# The platform picks up this file and registers it in MLflow.

import json

import joblib

joblib.dump(model, MODEL_OUTPUT_PATH)
print(f"Model saved to {MODEL_OUTPUT_PATH}")

# The exact frame passed to fit() (same columns, same order). The platform
# builds the model's test page from it: inputs, ranges and an example.
if TRAINING_FEATURES_PATH:
    X_train.to_parquet(TRAINING_FEATURES_PATH)

# What each input and each class means, for whoever uses the model.
NOTES = {
    "target": {"name": "species", "description": "Iris species"},
    "inputs": {
        "sepal_length": {"description": "Sepal length", "unit": "cm"},
        "sepal_width": {"description": "Sepal width", "unit": "cm"},
        "petal_length": {"description": "Petal length", "unit": "cm"},
        "petal_width": {"description": "Petal width", "unit": "cm"},
    },
    "classes": {str(i): name for i, name in enumerate(iris.target_names)},
}
if MODEL_NOTES_PATH:
    with open(MODEL_NOTES_PATH, "w", encoding="utf-8") as handle:
        json.dump(NOTES, handle, ensure_ascii=False)